In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as numpy
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
# Task 1: Write your code here:
food = os.path.join(path, 'Q1_data.csv')
data = pd.read_csv(food)

In [ ]:
# Task 2: Write your code here:
print(data.head())

In [ ]:
# Task 3: Write your code here:
print(data.info())

In [ ]:
# Task 4: Write your code here:
print(data.describe())

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(data['Delivery_Time'], bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('')
plt.show()

In [ ]:
# Task 1: Write your code here:
data = data.drop(columns=["Order_ID"])
data

In [ ]:
# Task 2: Write your code here:
missing_percentage = (data.isnull().sum() / len(data)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
categorical_cols = data.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
data.shape
# Weather, trafic_level, and Time_of_Day will be unkown since they are objects
# we will drop any Delivery_Time because we want a good prediction
# we will fill Courier_Experience_yrs with the average to minize the lost data

In [ ]:
data = data.dropna(subset=['Delivery_Time'])
data['Courier_Experience_yrs'] = data['Courier_Experience_yrs'].fillna(data['Courier_Experience_yrs'].mean())
for col in ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']:
    data[col] = data[col].fillna('unknown')
data.shape
#we didn't lose much data which is nice

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(data)
data.shape
#there was so many duplicates!!!

In [ ]:
# Task 4: Write your code here:

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

data.head()

In [ ]:
# Task 5: Write your code here:
features = data.drop(columns=['Delivery_Time'])
feature_cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
                'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
pd.DataFrame(features_scaled).head(3)

In [ ]:
# Task 6: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(data['Delivery_Time'], bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('delivery time')
plt.ylabel('')
plt.show()

# as shown in the graph below, the target is balanced

In [ ]:
# Task 1: Write your code here:
import numpy as np
X = features_scaled
y = data['Delivery_Time']
print(X)
print(y)

In [ ]:
# Task 2,3,4,5: Write your code here:
model = RandomForestRegressor(n_estimators=100)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))


# Print Evaluation Metrics
print(f"MAE : {np.mean(mae_scores):.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('delivery time estimation')
plt.xlabel('delivery time')
plt.ylabel('')
plt.show()

In [ ]:
# Task Bonus: Write your code here: